# 03: 标准化 + 高可变基因选择（批次感知）

在 02 合并后的 counts 矩阵上执行标准化管线，为后续 04 降维与 05 聚类准备数据。

**标准化**消除文库大小差异——不同细胞的测序深度天然不同，不做标准化会导致
高深度细胞主导后续所有分析。**高可变基因（HVG）选择**决定下游分析使用哪些基因——
只保留携带细胞类型差异信息的高可变基因，大幅降噪并节省内存。
**batch-aware HVG** 是整合分析的关键 trick：在每个批次（source_dataset）内独立选 HVG，
取跨批次共识——防止某个数据集的技术噪声基因霸占 HVG 列表。

**本 notebook 新增能力**（v2 科研工作台升级）：
- 标准化双模：经典 `normalize_total + log1p` 或 Pearson residuals（Lause 2021）
- HVG Scalar-or-Sweep 双模：写单值直接跑，写列表自动对比不同参数组合的 HVG 重叠度
- 批次感知 HVG：`batch_key` 指定按哪个列独立选 HVG
- HVG 排除列表：自动排除线粒体/核糖体/血红蛋白基因，避免技术性基因驱动聚类
- 可选回归混杂变量（`regress_out`）与缩放（`scale`），默认关闭

**本 notebook 产出**：
- `adata.layers['counts']` — 原始 counts，为下游 scVI / scANVI / DESeq2 保留
- `adata.X` — 标准化后的表达矩阵（float32）
- `adata.var['highly_variable']` — HVG 布尔掩码
- `adata.var['hvg_{flavor}_{n}']` — 各参数组合的 HVG 掩码（sweep 模式）
- `adata.uns['normalize_v1']` — 标准化参数记录
- 03 checkpoint `.h5ad` 文件，供 04 嵌入使用

In [ ]:
# === PARAMS ===

UPSTREAM_PATH = "results/02_merged_v1.h5ad"
OUTPUT_PATH   = "results/03_normalized_v1.h5ad"

# --- 标准化方法 ---
NORMALIZATION_METHOD = "standard"   # "standard" | "pearson_residuals"
                                    # standard = normalize_total + log1p（经典，适合大部分场景）
                                    # pearson_residuals = Lause 2021，更好的方差稳定化，跳过 log1p
TARGET_SUM = 1e4                    # normalize_total 的 target（仅 standard 模式生效）

# --- HVG 选择（支持 Scalar-or-Sweep 双模）---
# 写单值直接跑，写列表自动 sweep + 输出 HVG 集合重叠度对比
N_TOP_GENES = 2000                  # 单值 | 列表如 [1500, 2000, 3000, 4000]
HVG_FLAVOR  = "seurat"             # 单值 | 列表如 ["seurat", "seurat_v3"]

# --- 批次感知 HVG（多数据集整合的关键 trick）---
# 防止某个数据集的技术噪声基因霸占 HVG 列表
BATCH_AWARE_HVG = True
HVG_BATCH_KEY   = "source_dataset"  # 按哪个列做 batch-aware

# --- HVG 排除列表 ---
# 这些基因类别不应驱动聚类（它们有信息量但反映的是技术/通用状态而非细胞身份）
EXCLUDE_MT_FROM_HVG   = True        # 线粒体基因（MT-*）
EXCLUDE_RIBO_FROM_HVG = True        # 核糖体基因（RPS*/RPL*）
EXCLUDE_HB_FROM_HVG   = True        # 血红蛋白基因（HBA*/HBB*）
CUSTOM_EXCLUDE_PATTERNS = []        # 额外排除 pattern，如 ["^IG[HKL]", "^TR[ABGD]"]（免疫球蛋白/TCR）
EXCLUDE_CELL_CYCLE_FROM_HVG = False  # True=将 Tirosh 2015 S+G2M 基因加入排除列表（增殖信号主导 PCA 时启用）

# --- 回归混杂变量（可选，慎用）---
# 空列表 = 不回归。回归会 densify 矩阵、大幅增加内存和时间。
# 通常不推荐在此阶段回归——更好的做法是在 04 嵌入中通过 batch_key 处理。
# 仅在 PI 确认某个混杂因素严重干扰 HVG/PCA 时启用。
REGRESS_OUT = []                    # 可选：["pct_counts_mt", "total_counts", "S_score", "G2M_score"]

# --- 缩放（可选）---
# Harmony/scVI 不需要 scale；PCA-only 管线可能需要。
# 注意：scale 会 densify 矩阵！
SCALE = False
MAX_SCALE_VALUE = 10

OUTPUT_VERSION = 1
RANDOM_SEED    = 42

In [ ]:
# === Setup：sys.path + 导入依赖 ===
# sys.path 必须在 scanpy 导入之前设置，否则找不到 scrna_integration 模块。
import sys, os

_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# --- 所有 import 集中在这里 ---
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import gc
import re
import itertools

np.random.seed(RANDOM_SEED)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

In [ ]:
# === 加载上游 ===
print("Loading upstream:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 确认 adata.X 是原始 counts（上游 02 merged 应对齐此约定）
# 如果上游已做 transform，此处 dtype 会是 float 而非 int
if adata.X.dtype.kind == 'i':
    print("✓ adata.X 为整数 counts，符合上游合约")
else:
    print(f"⚠ adata.X dtype={adata.X.dtype}，非整数——上游可能已变换；确认是否需要调整上游管线")

## 保留原始 counts

标准化操作会改变 `adata.X` 中的原始计数数据。但下游方法（scVI、scANVI、
DESeq2、pseudobulk）需要原始整数计数来精准建模——因此先把 counts 拷贝到
`adata.layers['counts']` 妥善保存。

In [ ]:
# 将原始 counts 保留到 layers['counts']。
# 下游方法（scVI / scANVI / DESeq2 / pseudobulk）需要原始计数数据，
# 而非标准化后的数据。
print("Copying raw counts to adata.layers['counts']...")
adata.layers["counts"] = adata.X.copy()
print(f"layers keys: {list(adata.layers.keys())}")
print(f"counts layer dtype: {adata.layers['counts'].dtype}")

## 标准化

两种方法路线可选（通过 `NORMALIZATION_METHOD` 切换）：

- **`standard`**（默认）：`normalize_total` 将每个细胞缩放至相同总 UMI，消除测序深度差异；
  接着 `log1p` 做方差稳定化，将右偏分布拉近正态，使高表达基因不过度主导 PCA。
  这是单细胞领域的经典路线，适用绝大多数场景。
- **`pearson_residuals`**（Lause et al. 2021, Genome Biology）：基于负二项模型的残差变换，
  一步完成方差稳定化，不需要显式 log1p。优点是对 dropout 更鲁棒；
  缺点是不保留 log-normalized 空间的可解释性，且函数在 `sc.experimental` 中。

In [ ]:
# === 标准化分支 ===
if NORMALIZATION_METHOD == "standard":
    # normalize_total：每个细胞缩放至相同总 UMI，消除测序深度差异
    sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
    # log1p：log(1+x) 方差稳定化，使高表达基因不过度主导 PCA
    sc.pp.log1p(adata)
    print(f"✓ 标准化完成：normalize_total(target_sum={TARGET_SUM}) + log1p")
    print(f"  标准化后 X mean={adata.X.mean():.4f}, max={adata.X.max():.4f}")
elif NORMALIZATION_METHOD == "pearson_residuals":
    # Pearson residuals（Lause et al. 2021, Genome Biology）
    # 一步完成方差稳定化，不需要单独 log1p
    # 优点：对 dropout 更鲁棒；
    # 注意：sc.experimental API 可能在 scanpy 未来版本中变动
    sc.experimental.pp.normalize_pearson_residuals(adata)
    print("✓ 标准化完成：Pearson residuals (Lause 2021)")
    print(f"  标准化后 X mean={adata.X.mean():.4f}, std={adata.X.std():.4f}")
else:
    raise ValueError(f"不支持的 NORMALIZATION_METHOD: {NORMALIZATION_METHOD}，请使用 'standard' 或 'pearson_residuals'")

# 转为 float32——内存减半，单细胞数据有效精度无实质影响
adata.X = adata.X.astype(np.float32)
print(f"X dtype 已转为: {adata.X.dtype}")

## 标准化效果可视化

**看什么**：左图展示标准化前各细胞的总 UMI 计数分布（来自 counts layer）——
不同细胞测序深度差异悬殊。右图展示标准化后的表达值分布——
standard 模式下所有细胞缩放到统一尺度，pearson_residuals 模式下残差围绕 0 对称分布。

In [ ]:
# === 标准化前后对比图 ===
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：标准化前——从 counts layer 看每个细胞的总 UMI 分布
counts_before = np.array(adata.layers["counts"].sum(axis=1)).flatten()
axes[0].hist(counts_before, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(np.median(counts_before), color="red", linestyle="--",
                label=f'中位={np.median(counts_before):.0f}')
axes[0].set_xlabel("总 UMI 计数")
axes[0].set_ylabel("细胞数")
axes[0].set_title("标准化前\n各细胞总 UMI 计数差异大")
axes[0].legend(fontsize=9)

# 右图：标准化后——表达值分布
if sp.issparse(adata.X):
    sample_vals = adata.X.data[:200000] if len(adata.X.data) > 200000 else adata.X.data
else:
    sample_vals = adata.X.flatten()
axes[1].hist(sample_vals, bins=100, color="coral", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("标准化后表达值")
axes[1].set_ylabel("频数")
axes[1].set_title(f"标准化后（{NORMALIZATION_METHOD}）\n表达值分布")
axes[1].axvline(0, color="gray", linestyle=":", alpha=0.5)

plt.suptitle("标准化效果", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/03_normalize_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

# --- standard 模式额外诊断：top-3 基因标准化前后分布对比 ---
# 这是经典的 log1p 效果展示——看 log1p 如何将右偏分布拉近正态
if NORMALIZATION_METHOD == "standard":
    # 基于原始 counts 均值排名前 3 的基因
    top_idx = np.argsort(
        np.array(adata.layers["counts"].mean(axis=0)).flatten()
    )[-3:]
    top_genes = adata.var_names[top_idx].tolist()
    print(f"\ntop-3 基因表达分布对比: {top_genes}")

    # 重新计算 pre-log1p 标准化值（normalize_total 后、log1p 前）
    raw_top = adata.layers["counts"][:, top_idx].toarray()
    lib_size = np.array(adata.layers["counts"].sum(axis=1)).flatten()
    norm_expr = raw_top / lib_size[:, None] * TARGET_SUM

    # post-log1p 直接读取当前 adata.X
    log_expr = adata.X[:, top_idx].toarray()

    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
    for i, gene in enumerate(top_genes):
        axes2[0].hist(norm_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
        axes2[1].hist(log_expr[:, i], bins=50, alpha=0.5, label=gene, density=True)
    axes2[0].set_xlabel("标准化后表达量（未 log）")
    axes2[0].set_ylabel("密度")
    axes2[0].set_title("Log1p 前：高度右偏\n少数细胞极高值主导")
    axes2[0].legend(fontsize=8)
    axes2[1].set_xlabel("log1p 表达量")
    axes2[1].set_ylabel("密度")
    axes2[1].set_title("Log1p 后：接近正态\n适合 PCA 等线性方法")
    axes2[1].legend(fontsize=8)
    plt.suptitle("Log1p 方差稳定化效果", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/03_log1p_before_after.png", dpi=150, bbox_inches="tight")
    plt.show()

## 高可变基因选择（Scalar-or-Sweep 双模）

大多数基因在所有细胞中表达量相近（管家基因），不携带区分细胞类型的信息。
高可变基因（HVG）是那些在不同细胞间表达差异最大的基因——
它们携带了细胞类型、状态差异的核心信号。

**双模设计**：
- **单值模式**：`N_TOP_GENES=2000`，直接选一组 HVG，干净利落
- **Sweep 模式**：`N_TOP_GENES=[1500, 2000, 3000]`，自动遍历所有参数组合，
  输出 Jaccard 重叠度热力图，帮助 PI 判断参数敏感性

**批次感知 HVG**：当 `BATCH_AWARE_HVG=True` 时，scanpy 在每个 batch 内独立选 HVG，
取跨批次并集并按出现频率排名。这防止某个数据集的技术噪声基因因全局高变而被选中——
是多数据集整合的关键 trick。

In [ ]:
# === HVG 选择：Scalar-or-Sweep 双模 ===
# 写单值直接跑，写列表自动对比不同参数组合的 HVG 集合

_n_genes_values = N_TOP_GENES if isinstance(N_TOP_GENES, list) else [N_TOP_GENES]
_flavor_values = HVG_FLAVOR if isinstance(HVG_FLAVOR, list) else [HVG_FLAVOR]

hvg_results = {}  # 存储各组合的结果供对比

print(f"HVG Sweep: n_genes={_n_genes_values}, flavor={_flavor_values}")
print(f"BATCH_AWARE_HVG={BATCH_AWARE_HVG}", end="")
if BATCH_AWARE_HVG:
    print(f", batch_key={HVG_BATCH_KEY}")
else:
    print()
print()

for n_genes in _n_genes_values:
    for flavor in _flavor_values:
        label = f"n={n_genes}, flavor={flavor}"
        print(f"--- {label} ---")

        # batch-aware HVG：每个 batch 独立选 HVG，取并集按跨 batch 出现频率排名
        if BATCH_AWARE_HVG:
            if HVG_BATCH_KEY not in adata.obs.columns:
                raise KeyError(
                    f"HVG_BATCH_KEY='{HVG_BATCH_KEY}' 不在 obs 列中。"
                    f"可用列: {list(adata.obs.columns)[:10]}..."
                    f"请检查 BATCH_AWARE_HVG 和 HVG_BATCH_KEY 参数。"
                )
            sc.pp.highly_variable_genes(
                adata, n_top_genes=n_genes, flavor=flavor,
                batch_key=HVG_BATCH_KEY
            )
        else:
            sc.pp.highly_variable_genes(
                adata, n_top_genes=n_genes, flavor=flavor
            )

        # 存储结果到带参数后缀的 var 列，供后续对比
        key = f"hvg_{flavor}_{n_genes}"
        adata.var[key] = adata.var["highly_variable"].copy()
        n_selected = int(adata.var[key].sum())
        hvg_results[key] = {"n_genes": n_genes, "flavor": flavor, "n_selected": n_selected}
        print(f"  {key}: {n_selected} genes selected")

print(f"\n共生成 {len(hvg_results)} 组 HVG 结果")
print(f"adata.var['highly_variable'] 当前指向最后组合: {list(hvg_results.keys())[-1]}")

## HVG Sweep 对比

当使用列表模式（多个 `N_TOP_GENES` 或 `HVG_FLAVOR` 值）时，
下方 cell 输出不同参数组合间 HVG 集合的 Jaccard 相似度热力图。

**解读**：
- 对角线 = 1.0（自身完全一致）
- Jaccard 接近 1.0 的 pair → 参数选择对该对不敏感，可放心任选
- Jaccard 显著低于 1.0 的 pair → HVG 集合对参数敏感，建议 PI 目视下游 UMAP 效果后决定

单值模式下此 cell 仅打印确认信息，不画图。

In [ ]:
# === HVG Sweep 对比：Jaccard 重叠度热力图 ===
if len(hvg_results) > 1:
    # Jaccard 相似度矩阵：不同参数组合间 HVG 集合的重叠度
    keys = list(hvg_results.keys())
    jaccard_matrix = pd.DataFrame(index=keys, columns=keys, dtype=float)

    for k1, k2 in itertools.combinations(keys, 2):
        set1 = set(adata.var_names[adata.var[k1]])
        set2 = set(adata.var_names[adata.var[k2]])
        j = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0.0
        jaccard_matrix.loc[k1, k2] = j
        jaccard_matrix.loc[k2, k1] = j
    np.fill_diagonal(jaccard_matrix.values, 1.0)

    # 热力图
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(jaccard_matrix.values.astype(float), cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(keys)))
    ax.set_yticklabels(keys, fontsize=9)
    plt.colorbar(im, ax=ax, label="Jaccard similarity")
    ax.set_title("HVG 集合重叠度（不同参数组合间）", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/03_hvg_jaccard_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 汇总表
    print("\nHVG Sweep 汇总:")
    summary_df = pd.DataFrame(hvg_results).T
    from IPython.display import display as ipy_display
    ipy_display(summary_df)
else:
    entry = list(hvg_results.values())[0]
    print(f"✓ 单值模式：n_genes={entry['n_genes']}, flavor={entry['flavor']}, "
          f"选定 {entry['n_selected']} HVG")

## HVG 排除列表

某些基因类别表达量在不同细胞间确实高度可变，但这种可变性反映的是
**通用细胞状态**（如代谢活性、应激水平）或**技术背景**而非**细胞身份**——
它们不应驱动聚类和细胞类型鉴定。排除后依赖真正的细胞身份基因来区分细胞类型。

**排除类别**：
- **线粒体基因（MT-*）**：反映细胞膜破损/凋亡程度，是 QC 指标而非身份信号
- **核糖体基因（RPS*/RPL*）**：反映蛋白质合成活性，与细胞类型关联弱、批次间波动大
- **血红蛋白基因（HBA*/HBB*）**：红细胞污染标志，在组织样本中来自血液残留
- **自定义 pattern**：如免疫球蛋白（IG[HKL]）和 TCR（TR[ABGD]）基因——
  在淋巴细胞中高度可变，但反映的是克隆型而非细胞类型

**注意**：排除后**不做补位**。补位会把刚排除的基因类别中排名较低的成员重新拉回来，
削弱排除效果。如需精确某数量的 HVG，适当增大 `N_TOP_GENES` 初始值即可。

In [ ]:
# === HVG 排除列表：构建排除 mask，移除后不补位 ===
exclude_mask = pd.Series(False, index=adata.var_names)
excluded_counts = {}

if EXCLUDE_MT_FROM_HVG:
    mt_mask = adata.var_names.str.upper().str.startswith("MT-")
    excluded_counts["MT"] = int((adata.var["highly_variable"] & mt_mask).sum())
    exclude_mask |= mt_mask

if EXCLUDE_RIBO_FROM_HVG:
    ribo_mask = adata.var_names.str.upper().str.match("^(RPS|RPL)")
    excluded_counts["Ribo"] = int((adata.var["highly_variable"] & ribo_mask).sum())
    exclude_mask |= ribo_mask

if EXCLUDE_HB_FROM_HVG:
    hb_mask = adata.var_names.str.upper().str.match("^HB[^P]")  # HBA/HBB 但不含 HBP
    excluded_counts["HB"] = int((adata.var["highly_variable"] & hb_mask).sum())
    exclude_mask |= hb_mask

for pattern in CUSTOM_EXCLUDE_PATTERNS:
    try:
        custom_mask = adata.var_names.str.contains(pattern, case=False, regex=True)
    except re.error as e:
        print(f"⚠ CUSTOM_EXCLUDE_PATTERNS 中 '{pattern}' 正则无效 ({e})，跳过")
        continue
    excluded_counts[pattern] = int((adata.var["highly_variable"] & custom_mask).sum())
    exclude_mask |= custom_mask

# 细胞周期基因排除——当增殖信号主导 PCA 时启用
# Tirosh et al. 2015 的 S 期和 G2M 期 marker genes
if EXCLUDE_CELL_CYCLE_FROM_HVG:
    s_genes_cc = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
    g2m_genes_cc = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
    cc_genes = set(s_genes_cc + g2m_genes_cc)
    cc_mask = adata.var_names.isin(cc_genes)
    excluded_counts["CellCycle"] = int((adata.var["highly_variable"] & cc_mask).sum())
    exclude_mask |= cc_mask

# 移除
n_before = int(adata.var["highly_variable"].sum())
adata.var.loc[exclude_mask, "highly_variable"] = False
n_after = int(adata.var["highly_variable"].sum())
n_removed = n_before - n_after

print(f"HVG 排除列表：移除 {n_removed} 基因")
for cat, count in excluded_counts.items():
    if count > 0:
        print(f"  {cat}: {count}")
print(f"剩余 HVG: {n_after}")
print(f"（不做补位——如需精确数量请增大 N_TOP_GENES 初始值）")

## HVG 诊断图

**平均表达量 vs 离散度图**：每个点是一个基因。X 轴是平均表达量（log scale），
Y 轴是标准化离散度。蓝色点为选中的 HVG，灰色点为未选中。
理想情况下蓝色点应在各个表达水平上均匀覆盖高离散度区域。

**跨批次出现频率图**（batch-aware 模式专属）：展示 HVG 在多少个 batch 中被选中。
出现频率越高的基因越可能是跨数据集保守的真信号，而非单数据集的技术噪声。

In [ ]:
# === HVG 诊断图 ===
# 图1：平均表达量 vs 标准化离散度（标准 scanpy 诊断图）
sc.pl.highly_variable_genes(adata, show=True)
plt.savefig("results/figures/03_hvg_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()
print("HVG 诊断图已保存至 results/figures/03_hvg_diagnostic.png")

# 图2：batch-aware 时额外输出——per-batch HVG 出现频率分布
if BATCH_AWARE_HVG and "highly_variable_nbatches" in adata.var.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    nbatches_dist = adata.var["highly_variable_nbatches"].value_counts().sort_index()
    nbatches_dist.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel("出现在几个 batch 中")
    ax.set_ylabel("基因数")
    ax.set_title("HVG 跨 batch 出现频率\n（越多 batch 共享 = 越可能是真信号）")
    plt.tight_layout()
    plt.savefig("results/figures/03_hvg_nbatches.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # 统计摘要
    n_total_batches = adata.obs[HVG_BATCH_KEY].nunique()
    n_ubiquitous = int((adata.var["highly_variable_nbatches"] == n_total_batches).sum())
    print(f"总 batch 数: {n_total_batches}")
    print(f"在所有 {n_total_batches} 个 batch 中均为 HVG 的基因: {n_ubiquitous}")
elif BATCH_AWARE_HVG:
    print("（batch-aware 模式下未检测到 highly_variable_nbatches 列，可能 flavor 不支持此输出）")

## （可选）回归混杂变量

`sc.pp.regress_out` 用线性回归从表达矩阵中移除指定协变量的影响。

**为什么不推荐在此阶段回归？**
1. 回归会 **densify** 稀疏矩阵，内存占用可能膨胀 10-50 倍
2. 下游 Harmony/scVI 已内置批次/协变量处理，通常无需提前回归
3. 线性回归假设协变量与表达量是线性关系——对单细胞 counts 数据往往不成立

**什么时候启用？** PI 在 04 嵌入后发现某个混杂因素（如 MT% 或 total_counts）
严重主导了 PCA/UMAP 且 Harmony/scVI 无法充分校正时，才考虑回归。

In [ ]:
# === 回归混杂变量（可选）===
if REGRESS_OUT:
    print(f"⚠ 回归混杂变量：{REGRESS_OUT}")
    print("  注意：回归会 densify 矩阵，大幅增加内存。通常不推荐——")
    print("  更好的做法是在 04 嵌入中通过 batch_key/covariates 处理。")

    # 检查所有回归变量是否存在于 adata.obs
    missing = [v for v in REGRESS_OUT if v not in adata.obs.columns]
    if missing:
        raise KeyError(f"REGRESS_OUT 中包含不存在于 adata.obs 的列: {missing}。"
                       f"可用列: {list(adata.obs.columns)}")

    # 回归前的内存快照
    was_sparse = sp.issparse(adata.X)
    sc.pp.regress_out(adata, REGRESS_OUT)
    adata.X = adata.X.astype(np.float32)

    # 尝试恢复 sparse（如果足够稀疏）
    if not sp.issparse(adata.X):
        zero_fraction = (adata.X == 0).mean()
        if zero_fraction > 0.5:
            adata.X = sp.csr_matrix(adata.X)
            print(f"  回归后恢复 sparse（稀疏度 {zero_fraction:.1%}）")
        else:
            print(f"  ⚠ 回归后矩阵不够稀疏（零值占比 {zero_fraction:.1%}），保持 dense")
            print(f"    内存占用约 {adata.X.nbytes / 1024**2:.0f} MB")
else:
    print("✓ 跳过回归（REGRESS_OUT 为空）")

## （可选）缩放

`sc.pp.scale` 将每个基因的表达量标准化到均值为 0、方差为 1——
使 PCA 中各基因权重均等，避免高表达基因主导主成分。

**注意**：
- scale 会 **densify** 矩阵
- **Harmony/scVI 不需要 scale**——Harmony 在 PCA 空间校正、scVI 用原始 counts + 内部归一化
- PCA-only 管线（不做任何批次校正的简单分析）可能需要
- scale 前会保存 normalized 层到 `adata.layers["normalized"]`，以便后续可恢复

In [ ]:
# === 缩放（可选）===
if SCALE:
    print(f"缩放：max_value={MAX_SCALE_VALUE}")
    print("  注意：scale 会 densify 矩阵。Harmony/scVI 不需要 scale。")

    # 保存 normalized 到 layer 以便后续可恢复 sparse
    adata.layers["normalized"] = adata.X.copy()
    sc.pp.scale(adata, max_value=MAX_SCALE_VALUE)
    adata.X = adata.X.astype(np.float32)
    print(f"  缩放后 X mean={adata.X.mean():.4f}, std={adata.X.std():.4f}")
else:
    print("✓ 跳过缩放（SCALE=False，Harmony/scVI 不需要）")

## 参数记录 + Checkpoint

将标准化参数写入 `adata.uns` 以便追溯，执行内存自检确保数据完整性，
然后将产出写入磁盘形成 03 checkpoint。

**写入的 uns 字段**：
- `normalize_v1` — 本次运行的完整参数记录
- `stage` / `status` / `upstream` / `version` — 管线追溯链

**内存自检**：确认 `adata.X` 是 float32（sparse 或 dense 均可，
因 regress_out 或 scale 可能导致 dense——允许但警告）。

In [ ]:
# === 参数记录 + Checkpoint ===

# 记录运行参数
adata.uns["normalize_v1"] = {
    "method": NORMALIZATION_METHOD,
    "target_sum": TARGET_SUM if NORMALIZATION_METHOD == "standard" else None,
    "n_top_genes": N_TOP_GENES if not isinstance(N_TOP_GENES, list) else _n_genes_values[-1],
    "hvg_flavor": HVG_FLAVOR if not isinstance(HVG_FLAVOR, list) else _flavor_values[-1],
    "batch_aware": BATCH_AWARE_HVG,
    "hvg_batch_key": HVG_BATCH_KEY if BATCH_AWARE_HVG else None,
    "excluded_from_hvg": excluded_counts,
    "regress_out": REGRESS_OUT,
    "scale": SCALE,
}
print("normalize_v1:", adata.uns["normalize_v1"])

# 版本追踪——供迭代回跑追溯链使用
adata.uns["stage"] = "03_normalized"
adata.uns["status"] = "experimental"          # PI 审查后手动改为 "promoted"
adata.uns["upstream"] = [UPSTREAM_PATH]       # 上游文件，完整溯源链
adata.uns["version"] = f"v{OUTPUT_VERSION}"   # 与 OUTPUT_PATH 版本号一致

# --- 写入前检查 ---
if sp.issparse(adata.X):
    assert adata.X.dtype == np.float32, (
        f"dtype 应为 float32，实际 {adata.X.dtype}"
    )
    print("内存自检通过: X 是 sparse float32")
else:
    # regress_out 或 scale 可能导致 dense，允许但警告
    print(f"⚠ 矩阵为 dense（可能因 regress_out 或 scale），写入会较大")
    print(f"  内存占用约 {adata.X.nbytes / 1024**2:.0f} MB")
    assert adata.X.dtype == np.float32, (
        f"dtype 应为 float32，实际 {adata.X.dtype}"
    )

# --- 写入 checkpoint ---
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"✓ 写入 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes, "
      f"HVG={int(adata.var['highly_variable'].sum())})")

# 校验文件正确写出
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

# --- 释放内存 ---
# 跨 stage 边界释放内存，避免在同一 kernel 中累积
del adata
gc.collect()
print("Memory released.")